# 🏗 Feature Engineering
* Derive Features from existing Features:
    * Ratio Features ("Width_to_Height" etc.), with a log variant
    * Log Transformed Features
    * Polynomials
* Embed into (more or less) clean Custom Transformers that implement sklearn's TransformerMixin
* Create a (more or less) clean sklearn Preprocessor of Pipelines, ColumnTransformers, and FeatureUnions
* Test with some non fine-tuned default models

# Dependencies & Settings

In [ ]:
import math
from pathlib import Path
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import preprocessing, pipeline, compose, linear_model, model_selection, metrics, ensemble, neighbors
from sklearn.base import TransformerMixin
from sklearn.pipeline import FeatureUnion
import sklearn
import lightgbm as lgb

In [ ]:
from warnings import simplefilter
simplefilter("ignore", category=FutureWarning)

sklearn.set_config(transform_output = "pandas")

In [ ]:
%%html
<style>
table {float:left}
</style>

# Files

In [ ]:
df_train = pd.read_csv("/kaggle/input/playground-series-s4e4/train.csv")
print(f'{df_train.shape=}')

df_test = pd.read_csv("/kaggle/input/playground-series-s4e4/test.csv")
print(f'{df_test.shape=}')

df_original = pd.read_csv("/kaggle/input/abalone-dataset/abalone.csv")
print(f'{df_original.shape=}')

In [ ]:
RENAME_MAP = {
    'Whole weight': 'Whole_weight',
    'Whole weight.1': 'Shucked_weight',
    'Shucked weight': 'Shucked_weight',
    'Whole weight.2': 'Viscera_weight',
    'Viscera weight': 'Viscera_weight',
    'Shell weight': 'Shell_weight',
}

df_train = df_train.rename(columns=RENAME_MAP).drop('id', axis=1)
df_test = df_test.rename(columns=RENAME_MAP).drop('id', axis=1)
df_original = df_original.rename(columns=RENAME_MAP)

# make life easier for LGBM
df_train['Sex'] = df_train['Sex'].astype("category")  
df_test['Sex'] = df_test['Sex'].astype("category")  
df_original['Sex'] = df_original['Sex'].astype("category")  

In [ ]:
(df_original['Height'] < 0.1).sum()

# Merge for Training


In [ ]:
_df_merged = pd.concat([
    df_train,
    df_original
]).reset_index(drop=True)

df_features = _df_merged.drop(['Rings'], axis=1)
ser_targets = _df_merged['Rings']
print(f'{df_features.shape=}\n{ser_targets.shape=}')

assert not df_features.isna().any().any()

In [ ]:
metric_cols = ['Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']
categorical_cols = ['Sex']
assert set(df_features.columns) == set(metric_cols+categorical_cols)

# Ratio Features
Create features as simple proportion of feature A (numerator) to feature B (denominator). We can expect all feature values to be strictly positive, so don't have to worry about DivisionByZero. 

In [ ]:
class RatioFeaturesGenerator(TransformerMixin):
    """Create features as simple proportion of feature A (numerator) to feature B (denominator). Columns must be numerical."""

    def __init__(self, log=False): #cols: list[str], l):
#         self.cols = cols
        self.log = log

    def fit(self, X: pd.DataFrame, y=None):
        self.columns_ = [f'log_{col}' for col in X.columns]
        return self
    
    def fit(self, X: np.ndarray | pd.DataFrame, y=None):
#         self.minimum_value  # 0 will be overwritten with this to avoid infinities
        return self    

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if isinstance(X, pd.DataFrame):
            columns = list(X.columns)
            X = X.copy()
        elif isinstance(X, np.ndarray):
            columns = [f'feature_{i}' for i in range(X.shape[1])]
            X = pd.DataFrame(X, columns=columns)
        else:
            raise ValueError
        
        X_generated = pd.DataFrame()
        for i_numerator in range(len(columns)):
            col_numerator = columns[i_numerator]
            for col_denominator in columns[i_numerator+1:]:
                if self.log:
                    ratio_feature_name = f'log_ratio_{col_numerator}_to_{col_denominator}'
                    X_generated[ratio_feature_name] = np.log(X[col_numerator].values.clip(min=0.0001) / X[col_denominator].values.clip(min=0.0001))
                else:
                    ratio_feature_name = f'ratio_{col_numerator}_to_{col_denominator}'
                    X_generated[ratio_feature_name] = X[col_numerator] / X[col_denominator].values.clip(min=0.0001)
        
        self.columns_ = list(X_generated.columns)
        return X_generated
    
    def get_feature_names_out(self, *args, **params):
        """
        enable native DataFrame output after...
            sklearn.set_config(transform_output = 'pandas')
        """
        return self.columns_

In [ ]:
df_ratios = RatioFeaturesGenerator().fit_transform(df_features[metric_cols])
assert np.isinf(df_ratios).values.sum()  == 0
assert np.isnan(df_ratios).values.sum()  == 0
df_ratios

In [ ]:
df_log_ratios = RatioFeaturesGenerator(log=True).fit_transform(df_features[metric_cols])
assert np.isinf(df_log_ratios).values.sum()  == 0
assert np.isnan(df_log_ratios).values.sum()  == 0
df_log_ratios

# Log-Transform Features

In [ ]:
class LogTransformer(TransformerMixin):
    """Apply log transformation. Basically like sklearn's FunctionTransformer but returns meaningful column names"""

    def fit(self, X: np.ndarray | pd.DataFrame, y=None):
        if isinstance(X, np.ndarray):
            self.columns_ = [f'log_feature_{i}' for i in range(X.shape[1])]
        elif isinstance(X, pd.DataFrame):
            self.columns_ = [f'log_{col}' for col in X.columns]
        else:
            raise ValueError
        return self

    def transform(self, X: np.ndarray | pd.DataFrame | pd.DataFrame):
        assert X.shape[1] == len(self.columns_), "Bad Columns Number"
        X: np.ndarray = X if isinstance(X, np.ndarray) else X.values
        X_generated: np.ndarray = np.zeros((len(X), len(self.columns_)))
        
        for i in range(X.shape[1]):
            X_generated[:,i] = np.log1p(X[:,i].clip(min=0.0001))
        return X_generated
    
    def get_feature_names_out(self, *args, **params):
        """
        enable native DataFrame output after...
            sklearn.set_config(transform_output = 'pandas')
        """
        return self.columns_

In [ ]:
df_log_transformed = LogTransformer().fit_transform(df_features[metric_cols])
assert np.isinf(df_log_transformed).values.sum()  == 0
assert np.isnan(df_log_transformed).values.sum()  == 0

df_log_transformed

# Polynomial Features

In [ ]:
df_feature_polynomials = preprocessing.PolynomialFeatures(degree=2).fit_transform(df_features[metric_cols])
display(df_feature_polynomials.columns)
df_feature_polynomials

# Preprocessor 

In [ ]:
# Standardizing is useless after log transformation, so we add it only to the original features, polynomials and the ratio features
transformer_original = pipeline.Pipeline([
    ('scale', preprocessing.StandardScaler()),
])

transformer_ratio = pipeline.Pipeline([
    ('compute_ratio', RatioFeaturesGenerator()),
    ('scale', preprocessing.StandardScaler()),
])

transformer_poly = pipeline.Pipeline([
    ('compute_polynomials', preprocessing.PolynomialFeatures(degree=2)),
    ('scale', preprocessing.StandardScaler()),
])

In [ ]:
# use FeatureUnion to apply multiple transformations on original features
metric_fe = FeatureUnion([
    ("original", transformer_original),  # don't drop original features
    ("ratio", transformer_ratio),
    ("log_ratio", RatioFeaturesGenerator(log=True)),
    ("log_transform", LogTransformer()),
    ("polynomials", transformer_poly),
])
metric_fe

In [ ]:
# final preprocessor wrapping metric columns and 'Sex'
preprocessor = compose.ColumnTransformer(
    transformers=[
        ('metric', metric_fe, metric_cols),
        ('categorical', preprocessing.OneHotEncoder(sparse_output=False), categorical_cols)
    ])
preprocessor

In [ ]:
# Sanity Check
X = preprocessor.fit_transform(df_features)
print(f'{X.shape=}')

assert np.isinf(X).values.sum()  == 0
assert np.isnan(X).values.sum()  == 0

# Train GBDT with new Features
* default LGBM Regressor
* no fine-tuning
* 5-fold CV

# CV Scoring 

In [ ]:
def score_pipeline(pipeline: pipeline.Pipeline) -> float:
    
    arr_oof_predictions = np.zeros(len(ser_targets))
    kf = model_selection.KFold(
        n_splits=5, shuffle=True, random_state=42
    )
    for fold_no, (train_idx, val_idx) in enumerate(kf.split(df_features)):
        print(f"Starting fold {fold_no}")

        df_train_fold = df_features.iloc[train_idx].reset_index(drop=True)
        df_val_fold = df_features.iloc[val_idx].reset_index(drop=True)

        ser_targets_train_fold = ser_targets.iloc[train_idx].reset_index(drop=True)
        ser_targets_val_fold = ser_targets.iloc[val_idx].reset_index(drop=True)
        
        pipeline.fit(df_train_fold, ser_targets_train_fold)
        arr_oof_predictions[val_idx] = pipeline.predict(df_val_fold)

    arr_oof_predictions = arr_oof_predictions.clip(min=0)

    rmsle = metrics.mean_squared_log_error(y_true=ser_targets,
                                           y_pred=arr_oof_predictions,
                                           squared=False)  # True would get us the MSLE
    print(f'{rmsle=}')
    return rmsle

# LinearRegression

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', linear_model.LinearRegression())
])
pipe.set_output(transform="pandas")

display(pipe)

rmsle = score_pipeline(pipe)

# BayesianRidge

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', linear_model.BayesianRidge())
])
pipe.set_output(transform="pandas")

display(pipe)

rmsle = score_pipeline(pipe)

# HistGradientBoostingRegressor

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', ensemble.HistGradientBoostingRegressor())
])
pipe.set_output(transform="pandas")

display(pipe)

rmsle = score_pipeline(pipe)

# KNeighborsRegressor

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', neighbors.KNeighborsRegressor())
])
pipe.set_output(transform="pandas")

display(pipe)

rmsle = score_pipeline(pipe)

# LGBM Regressor
Note: LGBM does not like scaling, we still keep the preprocessing pipeline the same as above

In [ ]:
pipe = pipeline.Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', lgb.LGBMRegressor())
])
pipe.set_output(transform="pandas")

display(pipe)

rmsle = score_pipeline(pipe)

# Results
* All models score slightly better with the engineered features, cf. https://www.kaggle.com/stopwhispering/abalone-classic-ml-models
* Probably most of the engineered features should be discarded though. It requires some selection process.